# OneVoice V2 — Local runtime artifacts

Chuẩn bị các model đã đạt benchmark vào Google Drive và tạo `runtime_demo_local.yaml`. Notebook không fine-tune, không dùng INT8 SenseVoice failed và không thay đổi `config/config.yaml` trong Git.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys

GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
REPO = Path('/content/OneVoice')
ROOT = Path('/content/drive/MyDrive/OneVoice')
MODEL_ROOT = ROOT / 'models'
CACHE_ROOT = ROOT / 'model_cache'

if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', GITHUB_REPO, str(REPO)], check=True)
os.chdir(REPO)
os.environ['PYTHONUNBUFFERED'] = '1'
sys.path.insert(0, str(REPO / 'src'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'huggingface_hub', 'sherpa-onnx', 'funasr-onnx', 'PyYAML', 'soundfile', 'numpy==2.2.6', 'transformers==4.57.1', 'tokenizers==0.22.1', 'sentencepiece==0.2.0'], check=True)


In [ ]:
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from asr.asr_manager import GIPFORMER_INT8_FILES, GIPFORMER_REPO, GIPFORMER_REVISION

MT_VI2EN = MODEL_ROOT / 'envit5_finetuned_vi2en_v1'
MT_EN2VI = MODEL_ROOT / 'envit5_finetuned_en2vi_v1/best'
GIPFORMER = MODEL_ROOT / 'gipformer'
SENSEVOICE = MODEL_ROOT / 'sensevoice_en_construction_v1_onnx_fp32'

api = HfApi()
MT_VI2EN_SOURCE = 'platypus123/onevoice-envit5-vi-en'
MT_EN2VI_SOURCE = 'platypus123/onevoice-envit5-en-vi'
SENSEVOICE_SOURCE = 'platypus123/onevoice-sensevoice-en-construction-v1'
MT_VI2EN_REVISION = api.model_info(MT_VI2EN_SOURCE).sha
MT_EN2VI_REVISION = api.model_info(MT_EN2VI_SOURCE).sha
SENSEVOICE_REVISION = api.model_info(SENSEVOICE_SOURCE).sha
print('Syncing fine-tuned VI→EN model from Hugging Face...')
snapshot_download(MT_VI2EN_SOURCE, revision=MT_VI2EN_REVISION, local_dir=str(MT_VI2EN))

GIPFORMER.mkdir(parents=True, exist_ok=True)
for filename in GIPFORMER_INT8_FILES.values():
    target = GIPFORMER / filename
    if target.is_file():
        print('GIPFormer artifact already staged:', filename)
    else:
        hf_hub_download(GIPFORMER_REPO, filename, revision=GIPFORMER_REVISION, local_dir=str(GIPFORMER))

print('Syncing fine-tuned EN→VI model from Hugging Face...')
snapshot_download(MT_EN2VI_SOURCE, revision=MT_EN2VI_REVISION, local_dir=str(MT_EN2VI))
# SenseVoice runtime deliberately remains the locally exported ONNX bundle.
# The Hugging Face SenseVoice repository contains the training checkpoint,
# not the verified ONNX runtime bundle required by this config.
if not (SENSEVOICE / 'model.onnx').is_file():
    raise FileNotFoundError(
        f'Missing verified SenseVoice ONNX bundle on Drive: {SENSEVOICE}. '
        'Run colab_sensevoice_export_onnx_v1.ipynb first.'
    )
print('All required model directories are present.')


In [ ]:
REVIEWED_SAFETY_CSV = ROOT / 'review/safety_fast_path_review.csv'
SAFETY_MANIFEST = ROOT / 'artifacts/safety_audio_v1/manifest.json'
RUNTIME_CONFIG = ROOT / 'configs/runtime_demo_local.yaml'

command = [
    sys.executable, 'scripts/write_runtime_override.py',
    '--output', str(RUNTIME_CONFIG),
    '--gipformer-dir', str(GIPFORMER),
    '--sensevoice-dir', str(SENSEVOICE),
    '--mt-vi2en-dir', str(MT_VI2EN),
    '--mt-en2vi-dir', str(MT_EN2VI),
    '--safety-csv', str(REVIEWED_SAFETY_CSV),
    '--safety-manifest', str(SAFETY_MANIFEST),
    '--profile', 'development',
]
subprocess.run(command, check=True)
print(RUNTIME_CONFIG.read_text(encoding='utf-8'))


In [ ]:
import json
sys.path.insert(0, str(REPO / 'src'))
from safety.audio_store import SafetyAudioStore

store = SafetyAudioStore(SAFETY_MANIFEST, source_csv=REVIEWED_SAFETY_CSV)
audio, sample_rate = store.get('SAFE2_0001', 'vi2en')
assert audio.size > 0 and sample_rate > 0
print(f'Runtime artifact config ready: {RUNTIME_CONFIG}')
print(f'Safety fast-path smoke: SAFE2_0001/vi2en = {len(audio)} samples at {sample_rate} Hz')


## Hash-locked offline preflight

Tao manifest SHA-256 cho dung bundle tren Drive, gan no vao runtime config va kiem tra backend truoc startup. Cell nay chay CPU; lan dau co the mat vai phut vi phai hash model. `model_quant.onnx` bi loai tru vi INT8 da fail quality gate. License cua model upstream duoc danh dau can review, nen day la demo/dev preflight, khong phai xac nhan phat hanh production.


In [ ]:
from runtime.preflight import verify_artifacts

ARTIFACT_SPEC = ROOT / 'artifacts/runtime_demo_spec.json'
ARTIFACT_MANIFEST = ROOT / 'artifacts/runtime_demo_manifest.json'
RELEASE_LOCK = ROOT / 'artifacts/release_lock_v2.json'
GIPFORMER_LICENSE = 'MIT'
ENVIT5_LICENSE = 'OpenRAIL'
SENSEVOICE_LICENSE = 'Apache-2.0'
DEMO_LICENSE = 'IMPACT-INTERNAL-DEMO-NOT-FOR-REDISTRIBUTION'

command = [
    sys.executable, 'scripts/build_runtime_artifact_spec.py',
    '--output', str(ARTIFACT_SPEC),
    '--exclude-name', 'model_quant.onnx',  # INT8 candidate failed quality gate.
    '--asset', 'gipformer', str(GIPFORMER), 'vi2en', GIPFORMER_LICENSE,
    '--asset', 'mt_vi2en', str(MT_VI2EN), 'vi2en', ENVIT5_LICENSE,
    '--asset', 'sensevoice_fp32', str(SENSEVOICE), 'en2vi', SENSEVOICE_LICENSE,
    '--asset', 'mt_en2vi', str(MT_EN2VI), 'en2vi', ENVIT5_LICENSE,
    '--asset', 'safety_audio', str(ROOT / 'artifacts/safety_audio_v1'), 'vi2en,en2vi', DEMO_LICENSE,
    '--asset', 'reviewed_safety_csv', str(REVIEWED_SAFETY_CSV), 'vi2en,en2vi', DEMO_LICENSE,
]
subprocess.run(command, check=True)
subprocess.run([sys.executable, 'scripts/build_artifact_manifest.py', '--spec', str(ARTIFACT_SPEC), '--output', str(ARTIFACT_MANIFEST)], check=True)
subprocess.run([
    sys.executable, 'scripts/build_release_lock.py',
    '--artifact-manifest', str(ARTIFACT_MANIFEST), '--output', str(RELEASE_LOCK),
    '--release-id', 'onevoice-v2-rc1',
    '--model', 'gipformer_vi', GIPFORMER_REPO, GIPFORMER_REVISION, GIPFORMER_LICENSE, 'vi2en', 'gipformer',
    '--model', 'envit5_vi2en', MT_VI2EN_SOURCE, MT_VI2EN_REVISION, ENVIT5_LICENSE, 'vi2en', 'mt_vi2en',
    '--model', 'sensevoice_en', SENSEVOICE_SOURCE, SENSEVOICE_REVISION, SENSEVOICE_LICENSE, 'en2vi', 'sensevoice_fp32',
    '--model', 'envit5_en2vi', MT_EN2VI_SOURCE, MT_EN2VI_REVISION, ENVIT5_LICENSE, 'en2vi', 'mt_en2vi',
    '--safety-source', str(REVIEWED_SAFETY_CSV), '--safety-manifest', str(SAFETY_MANIFEST),
    '--safety-review-revision', 'impact-safety-v1',
], check=True)

# Rewrite only the Drive override, now pinned to the concrete artifact manifest.
command = [
    sys.executable, 'scripts/write_runtime_override.py',
    '--output', str(RUNTIME_CONFIG),
    '--gipformer-dir', str(GIPFORMER), '--sensevoice-dir', str(SENSEVOICE),
    '--mt-vi2en-dir', str(MT_VI2EN), '--mt-en2vi-dir', str(MT_EN2VI),
    '--safety-csv', str(REVIEWED_SAFETY_CSV), '--safety-manifest', str(SAFETY_MANIFEST),
    '--release-lock', str(RELEASE_LOCK), '--profile', 'development',
]
subprocess.run(command, check=True)

for direction in ('vi2en', 'en2vi'):
    result = verify_artifacts(RELEASE_LOCK, direction, 'development', sample_rate=16000)
    print(f'{direction}: verified {len(result["checked"])} local artifacts')
print(f'Offline config is hash-locked: {RUNTIME_CONFIG}')


## Offline safety end-to-end smoke

Dung 2 WAV safety da kiem checksum lam input o chieu nguoc lai. Nhieu hon lookup truc tiep: ASR phai nhan dung cau, Context Engine phai match dung safety ID va pipeline phai tra dung local target WAV. Khong can GPU hay network. Day la smoke voi audio demo tong hop; khong phai danh gia do ben ngoai cong truong.


In [ ]:
import numpy as np
import soundfile as sf

# The reviewed EN phrase "Stop immediately" resolves deterministically to
# SAFE2_0004 in the current approved CSV (other rows are Vietnamese variants).
SAFETY_ID = 'SAFE2_0004'
SMOKE_DIR = ROOT / 'reports/offline_safety_e2e'
payload = json.loads(SAFETY_MANIFEST.read_text(encoding='utf-8'))
entries = {(row['safety_id'], row['direction']): row for row in payload['entries']}

def safety_path(safety_id, direction):
    row = entries[(safety_id, direction)]
    path = Path(row['path'])
    return path if path.is_absolute() else SAFETY_MANIFEST.parent / path

# vi2en safety audio speaks English, so it is the EN input for en2vi; vice versa for Vietnamese.
cases = [('en2vi', 'vi2en'), ('vi2en', 'en2vi')]
for direction, input_audio_direction in cases:
    input_wav = safety_path(SAFETY_ID, input_audio_direction)
    expected_wav = safety_path(SAFETY_ID, direction)
    output_wav = SMOKE_DIR / f'{SAFETY_ID}_{direction}_output.wav'
    command = [
        sys.executable, 'src/pipeline.py', '--config', str(RUNTIME_CONFIG),
        '--direction', direction, '--profile', 'development', '--offline',
        '--input-file', str(input_wav), '--output-file', str(output_wav),
        '--report-dir', str(SMOKE_DIR),
    ]
    print('>', ' '.join(command), flush=True)
    subprocess.run(command, check=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    actual, actual_sr = sf.read(output_wav, dtype='float32')
    expected, expected_sr = sf.read(expected_wav, dtype='float32')
    assert actual_sr == expected_sr and actual.shape == expected.shape
    assert np.allclose(actual, expected, rtol=0, atol=1e-6), 'Output differs from verified safety WAV'
    print(f'PASS {direction}: ASR -> context -> {SAFETY_ID} -> verified local safety WAV')

print(f'Offline safety E2E smoke complete: {SMOKE_DIR}')
